<a href="https://colab.research.google.com/github/Pedro4Albuquerque/Processamento-de-Linguagem-Natural/blob/main/Aula08_Pratica_Passo_a_Passo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Instalação e download de recursos (rodar 1x)
!pip install -q spacy pandas
!python -m spacy download pt_core_news_sm -q

import spacy
import pandas as pd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 101.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
lexico_positivo = {
    "ótimo", "excelente", "adorar", "maravilhoso", "perfeito", "recomendar",
    "satisfeito", "rápido", "eficiente", "atencioso", "gentil", "confiável",
    "espaçoso", "iluminado", "tranquilo", "seguro", "bom"
}

lexico_negativo = {
    "péssimo", "horrível", "detestar", "decepcionado", "lento", "descaso",
    "problema", "sujo", "escuro", "barulhento", "caro", "insatisfeito",
    "atrasado", "ruim", "complicado", "incomodar"
}

print(f"Léxico positivo: {len(lexico_positivo)} palavras")
print(f"Léxico negativo: {len(lexico_negativo)} palavras")

Léxico positivo: 17 palavras
Léxico negativo: 16 palavras


In [13]:
def polaridade_simples(texto):
    doc = nlp(texto)
    positivas = sum(1 for token in doc if token.lemma_.lower() in lexico_positivo)
    negativas = sum(1 for token in doc if token.lemma_.lower() in lexico_negativo)

    if positivas > negativas:
        classificacao = "Positivo"
    elif negativas > positivas:
        classificacao = "Negativo"
    else:
        classificacao = "Neutro"

    return classificacao, positivas, negativas

In [14]:
import spacy

nlp = spacy.load("pt_core_news_sm")

corpus_reviews = [
    "O atendimento da corretora foi excelente, super atenciosa e rápida para responder.",
    "O apartamento é espaçoso e muito bem iluminado, adorei!",
    "Fiquei decepcionado com o atraso na entrega das chaves.",
    "O imóvel estava sujo e mal cuidado na visita.",
    "Achei o preço um pouco caro para a região.",
    "O corretor foi muito gentil e esclareceu todas as minhas dúvidas.",
    "A negociação foi tranquila e o processo foi rápido.",
    "O bairro é barulhento à noite, o que me incomodou.",
    "O apartamento tem dois quartos e uma vaga de garagem.",
    "Recomendo a imobiliária, atendimento excelente do início ao fim!",
]

for i, texto in enumerate(corpus_reviews, 1):
    classificacao, pos, neg = polaridade_simples(texto)
    print(f"Review {i}: {classificacao} (positivas={pos}, negativas={neg})")
    print(f"  \"{texto}\"")

Review 1: Positivo (positivas=3, negativas=0)
  "O atendimento da corretora foi excelente, super atenciosa e rápida para responder."
Review 2: Positivo (positivas=3, negativas=0)
  "O apartamento é espaçoso e muito bem iluminado, adorei!"
Review 3: Neutro (positivas=0, negativas=0)
  "Fiquei decepcionado com o atraso na entrega das chaves."
Review 4: Negativo (positivas=0, negativas=1)
  "O imóvel estava sujo e mal cuidado na visita."
Review 5: Negativo (positivas=0, negativas=1)
  "Achei o preço um pouco caro para a região."
Review 6: Positivo (positivas=1, negativas=0)
  "O corretor foi muito gentil e esclareceu todas as minhas dúvidas."
Review 7: Positivo (positivas=2, negativas=0)
  "A negociação foi tranquila e o processo foi rápido."
Review 8: Negativo (positivas=0, negativas=2)
  "O bairro é barulhento à noite, o que me incomodou."
Review 9: Neutro (positivas=0, negativas=0)
  "O apartamento tem dois quartos e uma vaga de garagem."
Review 10: Positivo (positivas=1, negativas=0)


In [15]:
doc_review3 = nlp(corpus_reviews[2])
for token in doc_review3:
    print(f"{token.text:15} -> lema: {token.lemma_}")

Fiquei          -> lema: Fiquei
decepcionado    -> lema: decepcionar
com             -> lema: com
o               -> lema: o
atraso          -> lema: atraso
na              -> lema: em o
entrega         -> lema: entrega
das             -> lema: de o
chaves          -> lema: chave
.               -> lema: .


In [16]:
lexico_negativo_v2 = lexico_negativo | {"decepcionar"}  # adicionando o lema correto

def polaridade_v2(texto):
    doc = nlp(texto)
    positivas = sum(1 for token in doc if token.lemma_.lower() in lexico_positivo)
    negativas = sum(1 for token in doc if token.lemma_.lower() in lexico_negativo_v2)
    if positivas > negativas:
        return "Positivo", positivas, negativas
    elif negativas > positivas:
        return "Negativo", positivas, negativas
    return "Neutro", positivas, negativas

print("Antes:", polaridade_simples(corpus_reviews[2]))
print("Depois:", polaridade_v2(corpus_reviews[2]))

Antes: ('Neutro', 0, 0)
Depois: ('Negativo', 0, 1)


In [17]:
frase_negacao = "O atendimento não foi bom."

print(polaridade_v2(frase_negacao))


('Positivo', 1, 0)


In [18]:
doc_negacao = nlp(frase_negacao)
for token in doc_negacao:
    print(f"{token.text:12} lema={token.lemma_:10} dep={token.dep_:10} head={token.head.text}")

O            lema=o          dep=det        head=atendimento
atendimento  lema=atendimento dep=nsubj      head=bom
não          lema=não        dep=advmod     head=bom
foi          lema=ser        dep=cop        head=bom
bom          lema=bom        dep=ROOT       head=bom
.            lema=.          dep=punct      head=bom


In [19]:
def polaridade_com_negacao(texto):
    doc = nlp(texto)
    positivas = 0
    negativas = 0

    for token in doc:
        lema = token.lemma_.lower()
        # Verifica se algum "filho" do token é uma negação ligada a ele (advmod)
        negado = any(
            filho.lemma_.lower() in ("não", "nunca", "jamais") and filho.dep_ == "advmod"
            for filho in token.children
        )

        if lema in lexico_positivo:
            if negado:
                negativas += 1
            else:
                positivas += 1
        elif lema in lexico_negativo_v2:
            if negado:
                positivas += 1
            else:
                negativas += 1

    if positivas > negativas:
        return "Positivo", positivas, negativas
    elif negativas > positivas:
        return "Negativo", positivas, negativas
    return "Neutro", positivas, negativas

print("Sem tratar negação:", polaridade_v2(frase_negacao))
print("Com tratamento:    ", polaridade_com_negacao(frase_negacao))

Sem tratar negação: ('Positivo', 1, 0)
Com tratamento:     ('Negativo', 0, 1)
